# PHYS 405 Computerized Data Analysis · Week 1 · Program and data structures
Instructor: Prof. Dr. Bora Işıldak · Department of Physics, Yeditepe University

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/Phys_405_2026_2027_Fall/blob/main/Labs/Labs1/Lab1.ipynb)

**File → Save a copy in Drive, then work in your copy.** Your work lives in your Drive copy, not in the link.

How the lab works (eight problems, about 14 minutes each, in pairs):
1. Read the problem. Write your prediction in the **Predict** cell.
2. Run the given code, then **Compare** with your prediction.
3. Run the checks.
4. Do the **Your turn** task in the empty cell.

The lab uses new data: the lecture's answers do not carry over, so make your predictions fresh.
Rule for the whole lab: predict before you run. A wrong prediction you wrote down is worth more than a right one you did not.

In [ ]:
import sys, math, time, warnings, numpy as np, matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'   # sharp figures on high-resolution screens
print("Python    ", sys.version.split()[0])
print("NumPy     ", np.__version__)
print("getsizeof(1.0) =", sys.getsizeof(1.0), "bytes")

## Problem 1 · Can a Julian date in float64 days time a 10 µs event?
Astronomers store times as Julian dates in days: today, 2026-10-05 00:00 UT, is JD 2461318.5. Two events 1 ms, 100 µs and 10 µs apart are stored as float64 JDs. What differences come back, in seconds?

*Given:* 1 day = 86400 s; the spacing of doubles is 2^(e−52) with 2^e ≤ x < 2^(e+1); a stored value is rounded to the nearest grid point; 2²¹ = 2 097 152, 2²² = 4 194 304.

**Predict.** Give the three differences (s). Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
jd = 2461318.5   # JD of 2026-10-05 00:00 UT, days
for dt in [1e-3, 1e-4, 1e-5]:   # s
    print(((jd + dt / 86400) - jd) * 86400)

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: independent method: the Modified Julian Date has a far finer grain
mjd = jd - 2400000.5
print(f"MJD = {mjd}: spacing {np.spacing(mjd) * 86400 * 1e6:.2f} us")
for dt in [1e-3, 1e-4, 1e-5]:
    print(f"  MJD: true {dt:.0e} s -> stored {((mjd + dt / 86400) - mjd) * 86400:.4e} s")
# Check B: magnitude by hand: 2^-31 d x 86400 s/d
print(f"JD spacing = {np.spacing(jd):.3e} d = {np.spacing(jd) * 86400 * 1e6:.1f} us")
print(f"2^-31 d = {2.0**-31 * 86400 * 1e6:.1f} us")
print(f"trap: eps x 1 day = {2.0**-52 * 86400:.1e} s is NOT the resolution of a JD")
assert abs(np.spacing(jd) * 86400 * 1e6 - 40.2) < 0.05, "the JD grain must be about 40 us"

**Your turn.** Unix time is seconds since 1970 in float64. Find the smallest power of two t = 2ᵏ s at which `np.spacing(t)` first exceeds 1 µs (a while loop that doubles t). Convert t to years.

*Hint:* start from `t = 1.0` and double while `np.spacing(t) <= 1e-6`; one year ≈ 3.156×10⁷ s.

In [ ]:
# TODO: write your answer here

## Problem 2 · A simulation clock that runs fast
A pendulum simulation advances time with `t += dt`, dt = 0.01 s, for 10⁵ steps (1000 s). With t stored as float32, what does t read at the end? And with float64?

*Given:* float32 keeps 24 significant bits (ε₃₂ = 1.19×10⁻⁷); the spacing of float32 near 1000 is 6.1×10⁻⁵ s.

**Predict.** Give the float32 and float64 readings (s). Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
t32, t64 = np.float32(0), 0.0
for _ in range(100_000):
    t32 += np.float32(0.01)
    t64 += 0.01
print(t32, t64)
print(f"float32: {float(t32):.4f} s (error {float(t32) - 1000:+.4f} s)")
print(f"float64: {t64:.10f} s (error {t64 - 1000:+.1e} s)")

*(The loop takes a few seconds in Colab.)*

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: independent method: t = n*dt in float32 at n = 1e5
print(f"t = n*dt in float32: {float(np.float32(100_000) * np.float32(0.01)):.4f} s;"
      f" spacing(float32, 1000) = {float(np.spacing(np.float32(1000))):.2e} s")
print(f"trap: float32(0.01) = {float(np.float32(0.01)):.12f}; its error {float(np.float32(0.01)) - 0.01:.2e} s"
      f" x 1e5 steps = {(float(np.float32(0.01)) - 0.01) * 1e5:.2e} s (not the main effect)")

# Check B: limiting case: dt = 2^-7 s is exact in binary, so are all the sums
def accumulate(dt, nsteps):
    t, d = np.float32(0), np.float32(dt)
    for _ in range(nsteps):
        t += d
    return float(t)
t_exact = accumulate(2.0**-7, 128_000)
print("dt = 2^-7 s, 128000 float32 steps ->", t_exact, "s")
assert t_exact == 1000.0, "a step that is a power of two must accumulate exactly"

**Your turn.** Rerun the float32 clock with dt = 0.1 s for 10⁴ steps, and with dt = 2⁻⁷ s = 0.0078125 s for 128 000 steps. Which drifts, and why does the second not drift at all?

*Hint:* 2⁻⁷ needs 1 significant bit; every partial sum k·2⁻⁷ up to 1000 fits in 24 bits.

In [ ]:
# TODO: write your answer here

## Problem 3 · Sixteen frames of 40 counts
A camera stores pixels as uint8. You add 16 dark frames of 40 counts each into `total = np.zeros_like(frame)` with `total += frame`. What does one pixel of `total` read?

*Given:* uint8 holds 0…255; the in-place `+=` keeps the dtype of `total`.

**Predict.** Give the value of one pixel. Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
frame = np.full((2, 2), 40, dtype=np.uint8)
total = np.zeros_like(frame)
for _ in range(16):
    total += frame
print(total[0, 0])

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: bound: uint8 holds at most 255
print("frames before overflow: 255 // 40 =", 255 // 40, "(7 x 40 = 280 > 255)")
# Check B: independent method: an int64 accumulator and np.sum (which upcasts)
total64 = np.zeros(frame.shape, dtype=np.int64)
for _ in range(16):
    total64 += frame
stack = np.stack([frame] * 16)
print("int64 accumulator:", total64[0, 0])
print("np.sum(stack, axis=0):", np.sum(stack, axis=0)[0, 0])
assert total64[0, 0] == 640 == np.sum(stack, axis=0)[0, 0], "the true sum of 16 x 40 is 640"
assert total[0, 0] == 640 % 256, "uint8 arithmetic is modulo 256"

**Your turn.** A 12-bit ADC gives readings 0…4095, stored as int16. You sum ten readings of 4000 in an int16 accumulator. What do you get, and which dtype should the accumulator have?

*Hint:* int16 range −32768…32767; 40000 − 65536.

In [ ]:
readings = np.full(10, 4000, dtype=np.int16)
# TODO: write your answer here

## Problem 4 · A one-kilometre baseline measured to micrometres
Six interferometer readings of a 1 km baseline: 1000 m + {3.1, 2.7, 3.4, 2.9, 3.2, 3.0} µm (illustrative). Before computing: how many digits does `mean(x²) − mean(x)²` lose? Then compute σ² both ways.

*Given:* x ≈ 10³ m, so x² ≈ 10⁶ m²; σ ≈ 0.2 µm, so σ² ≈ 5×10⁻¹⁴ m²; digits lost ≈ log₁₀(x²/σ²); the spacing of doubles near 10⁶ is 1.16×10⁻¹⁰.

**Predict.** Write the number of digits lost and both values of σ² (m²). Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
d_um = [3.1, 2.7, 3.4, 2.9, 3.2, 3.0]                  # deviations, um
x = 1000.0 + np.array(d_um) * 1e-6                     # m

def var_naive(x):                                      # one pass: mean of squares minus square of the mean
    n, s, s2 = len(x), 0.0, 0.0
    for xi in x:
        s += xi; s2 += xi * xi
    return s2 / n - (s / n) ** 2

def var_two_pass(x):                                   # subtract the mean first
    n = len(x); m = sum(x) / n
    return sum((xi - m) ** 2 for xi in x) / n

print(f"{np.mean(x**2) - np.mean(x)**2:.3e}")
print(f"{np.var(x):.4e}")
print(f"var_naive    = {var_naive(x):.3e} m^2")
print(f"var_two_pass = {var_two_pass(x):.4e} m^2;  sigma = {math.sqrt(var_two_pass(x)) * 1e6:.3f} um")
print(f"mean(x^2) ~ {np.mean(np.square(x)):.6e} m^2; spacing there = {np.spacing(1e6):.3e} m^2")
print(f"digits lost ~ log10(1e6/4.9e-14) = {math.log10(1e6 / 4.917e-14):.1f}")

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: sign: a variance is never negative
try:
    assert var_naive(x) >= 0, "a variance is never negative: the naive value is meaningless"
except AssertionError as e:
    print("AssertionError:", e, f"({var_naive(x):.3e} m^2)")
# Check B: independent method: np.var, and the naive formula on x - x0
print(f"np.var = {np.var(x):.4e} m^2")
print(f"naive on x - x0 = {var_naive([xi - x[0] for xi in x]):.4e} m^2")
print(f"naive on the um deviations = {var_naive(d_um):.4f} um^2")
assert var_naive([xi - x[0] for xi in x]) >= 0, "the shifted formula must not be negative"

**Your turn.** Implement Welford's update for a data stream: for k = 1, 2, …: δ = x_k − m; m ← m + δ/k; M₂ ← M₂ + δ(x_k − m); σ² = M₂/n. Run it on the six baseline readings.

*Hint:* use the updated m in the last factor; compare with `np.var` to 4 significant figures.

In [ ]:
x = 1000.0 + np.array([3.1, 2.7, 3.4, 2.9, 3.2, 3.0]) * 1e-6   # m
# TODO: write your answer here

## Problem 5 · The best step for ln x at x = 100
Compute the derivative of f(x) = ln x at x = 100 (exact 0.01) with the forward difference for h = 10⁻¹, 10⁻², …, 10⁻¹⁵. Predict the best h first from the error model of the lecture. Is it 10⁻⁸ again?

*Given:* E(h) ≈ ½h|f″| + 2ε|f|/h; here |f| = ln 100 = 4.605 and |f″| = 1/x² = 10⁻⁴; ε = 2.22×10⁻¹⁶.

**Predict.** Give the best h (compute h_opt = 2√(ε|f|/|f″|) by hand first). Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
eps = 2.0**-52
ks = np.arange(1, 16)
h = 10.0**-ks                                                     # step, dimensionless (x = 100)
err = np.array([abs((math.log(100 + hk) - math.log(100)) / hk - 0.01) for hk in h])
for k in (1, 3, 5, 7, 9, 12):
    print(f"h = 1e-{k:02d}: forward error {err[k - 1]:.2e}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(h, err, "o", label="measured |error|")
ax.loglog(h, 0.5 * h * 1e-4, "-", label="truncation: 1/2 h |f''| = 5e-5 h")
ax.loglog(h, 2 * eps * math.log(100) / h, "--", label="roundoff bound: 2 eps |f| / h")
ax.set_xlabel("step h (dimensionless, x = 100)"); ax.set_ylabel("error of the derivative (1/x units)")
ax.legend()
plt.show()

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: theory: h_opt from the model lies next to the best grid step
best = ks[np.argmin(err)]
h_opt = 2 * math.sqrt(eps * math.log(100) / 1e-4)
print(f"best grid h = 1e-{best} (error {err[best - 1]:.2e}); h_opt = {h_opt:.1e}")
assert best == 5, "the best grid step for ln x at 100 is 1e-5, not 1e-8"
# Check B: scaling: on the truncation side two decades of h give two decades of error
print(f"error(1e-1) / error(1e-3) = {err[0] / err[2]:.1f}")
assert abs(err[0] / err[2] - 100) < 1, "truncation error is proportional to h"

**Your turn.** Use the central difference [f(h) − f(−h)]/(2h) for f = eˣ at x = 0 (exact derivative 1) with h = 10⁻³, 10⁻⁵, 10⁻⁷, 10⁻⁹. Which is best? Explain with the central-difference error model.

*Hint:* truncation h²|f‴|/6, roundoff ε|f|/h; h_opt ~ ε^(1/3).

In [ ]:
# TODO: write your answer here

## Problem 6 · What do 10⁶ complex amplitudes cost?
A spectrometer gives 10⁶ complex amplitudes. Estimate the memory as a Python list of complex numbers and as a complex128 array. Then: what are the strides of `A[:, ::2]` for `A = np.zeros((3, 4))`, and of a (1000, 3) table of positions?

*Given:* `sys.getsizeof(1j)` = 32 bytes; a list stores 8-byte pointers; complex128 = 16 bytes; C order.

**Predict.** Give both memories (MB) and both strides. Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
n = 10**6
A = np.zeros((3, 4))
print(sys.getsizeof(1j), np.zeros(n, complex).nbytes)
print(A[:, ::2].strides, np.zeros((1000, 3)).strides)

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: independent method: the list is a pointer block plus n objects of 32 bytes
lst = [complex(i, 1) for i in range(n)]
list_mb = (sys.getsizeof(lst) + n * sys.getsizeof(1j)) / 1e6
arr = np.array(lst)
print(f"list = {list_mb:.1f} MB; complex128 array = {arr.nbytes / 1e6:.0f} MB; ratio = {list_mb * 1e6 / arr.nbytes:.1f}")
assert abs(list_mb - 40.4) < 0.05, "the list model gives 40.4 MB"
# Check B: view check: the slice shares memory with A
print("np.shares_memory(A, A[:, ::2]) =", np.shares_memory(A, A[:, ::2]))
assert np.shares_memory(A, A[:, ::2]), "a slice is a view, not a copy"

**Timing on your own machine.** No expected numbers: the ratios depend on the Colab machine. Record them.

In [ ]:
t0 = time.perf_counter(); s_list = sum(lst); t_list = time.perf_counter() - t0
t0 = time.perf_counter(); s_arr = arr.sum();  t_arr = time.perf_counter() - t0
print(f"sum(list) = {t_list * 1e3:.1f} ms; arr.sum() = {t_arr * 1e3:.2f} ms; ratio = {t_list / t_arr:.0f}")

M = np.ones((2000, 2000))
t0 = time.perf_counter(); r = [M[i, :].sum() for i in range(2000)]; t_row = time.perf_counter() - t0
t0 = time.perf_counter(); c = [M[:, j].sum() for j in range(2000)]; t_col = time.perf_counter() - t0
print(f"row sums = {t_row * 1e3:.1f} ms; column sums = {t_col * 1e3:.1f} ms; ratio = {t_col / t_row:.1f}")

**Your turn.** A temperature field on a 100 × 100 × 100 grid is stored as float64 in C order. Give its strides and memory, and say which index you should loop over innermost.

*Hint:* the last index changes fastest; each stride is 8 bytes times the sizes of the later axes.

In [ ]:
# TODO: write your answer here

## Problem 7 · A new detector run: how many particles?
The cell below simulates a new run (seed 2028, 3000 samples) with exactly 15 pulses. Count upward crossings of 0.20 V with a loop and with an array expression; then add a 30 µs dead time; then scan the threshold. Where is the count trustworthy?

*Given:* samples every 1 µs; τ = 8 µs; noise σ = 0.03 V; amplitudes 0.32…0.91 V; `count_loop` and `count_dead` as in the lecture.

**Predict.** Give the counts at 0.20 V (loop, array, dead time) and the range of thresholds where you trust the count. Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
def simulate(seed, n_pulses, n_samples, sigma=0.03, tau=8.0):
    rng = np.random.default_rng(seed)
    v = rng.normal(0.0, sigma, n_samples)                                   # noise, V
    starts = np.sort(rng.choice(np.arange(50, n_samples - 100, 80), n_pulses, replace=False))
    amps = rng.uniform(0.3, 1.0, n_pulses)                                  # V
    t = np.arange(n_samples)                                                # us
    for a, s in zip(amps, starts):
        v[t >= s] += a * np.exp(-(t[t >= s] - s) / tau)
    return v, amps

def count_loop(v, thr):
    c = 0
    for i in range(1, len(v)):
        if v[i - 1] < thr and v[i] >= thr:
            c += 1
    return c

def count_dead(v, thr, dead=30):
    c, last = 0, -10**9
    for i in range(1, len(v)):
        if v[i - 1] < thr <= v[i] and i - last > dead:
            c += 1; last = i
    return c

v, amps = simulate(2028, 15, 3000)
print(f"seed 2028: {len(amps)} pulses, {len(v)} samples, amplitudes {amps.min():.2f}..{amps.max():.2f} V;"
      f" smallest two {[float(q) for q in sorted(np.round(amps, 2))[:2]]}")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(np.arange(len(v)), v, lw=0.7)
ax.axhline(0.20, color="r", ls="--", label="threshold 0.20 V")
ax.set_xlabel("time (us)"); ax.set_ylabel("voltage (V)")
ax.legend()
plt.show()

print(count_loop(v, 0.20))
print(np.count_nonzero((v[:-1] < 0.20) & (v[1:] >= 0.20)))
print(count_dead(v, 0.20, dead=30))

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: robustness: scan the threshold, with and without dead time
cross = lambda v, thr: int(np.count_nonzero((v[:-1] < thr) & (v[1:] >= thr)))
scan = [0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
print("crossings: " + ", ".join(f"{q:.2f}:{cross(v, q)}" for q in scan))
print("dead time: " + ", ".join(f"{q:.2f}:{count_dead(v, q)}" for q in scan))
plateau = [q for q in scan if count_dead(v, q) == 15]
print("plateau (dead time) =", plateau)
assert plateau == [0.1, 0.12, 0.15, 0.2, 0.25, 0.3], "the dead-time count must plateau from 0.10 V to 0.30 V"

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(scan, [cross(v, q) for q in scan], "o-", label="crossings")
ax.plot(scan, [count_dead(v, q) for q in scan], "s-", label="with 30 us dead time")
ax.axhline(15, color="gray", ls=":", label="truth: 15")
ax.set_xlabel("threshold (V)"); ax.set_ylabel("count")
ax.legend()
plt.show()
# Check B: known truth: the simulation has 15 pulses
assert len(amps) == 15, "the simulation has 15 pulses"
print("truth:", len(amps))

**Your turn.** At 0.20 V, scan the dead time: 0, 5, 10, 20, 30, 60, 100 µs. Where is the count right, and what goes wrong at each end?

*Hint:* a pulse decays to e⁻⁴ ≈ 2 % after 4τ = 32 µs; pulses are at least 80 µs apart.

In [ ]:
# TODO: write your answer here

## Problem 8 · A pendulum file with a −999 in it
A group's raw file has two comment lines, a header, 20 rows (5 lengths × 4 trials), one missing period logged as −999 and one period typed in milliseconds. Read it with `genfromtxt` and compute g = 4π²L/⟨T⟩² for each length. Find both problems before you average anything.

*Given:* small-angle T = 2π√(L/g): about 0.9 s at 0.20 m, 1.27 s at 0.40 m, 2.0 s at 1.00 m; L = 0.20, 0.40, 0.60, 0.80, 1.00 m (illustrative data).

**Predict.** Say whether the file reads fine, and what ⟨T⟩ at 0.80 m would be if you average the raw column (s). Write your prediction in the next cell *before* you run anything below. It is a variable, so it is saved with the notebook.

In [ ]:
prediction = ...  # write your prediction before running the next cell

In [ ]:
raw = '''# pendulum, group B, illustrative data
# columns: length (m), trial number, period (s)
length_m,trial,period_s
0.20,1,0.888
0.20,2,0.900
0.20,3,0.874
0.20,4,0.914
0.40,1,1.276
0.40,2,1265
0.40,3,1.265
0.40,4,1.272
0.60,1,1.551
0.60,2,1.551
0.60,3,1.563
0.60,4,1.560
0.80,1,1.794
0.80,2,1.793
0.80,3,-999
0.80,4,1.787
1.00,1,2.001
1.00,2,2.013
1.00,3,2.005
1.00,4,1.990
'''
with open("pendulum.csv", "w") as f:
    f.write(raw)

data = np.genfromtxt("pendulum.csv", delimiter=",", names=True, skip_header=2)
L = data["length_m"]                 # m
T = data["period_s"]                 # s
print(len(T), "rows")
print(np.count_nonzero(T == -999), T[T > 10])
print(f"raw mean at 0.80 m = {T[L == 0.80].mean():.2f} s")
print(f"without the unit fix: g(0.40 m) = {4 * math.pi**2 * 0.40 / T[L == 0.40].mean()**2:.2e} m/s^2")

**Compare.** Put your `prediction` next to the output above. Was it right?
If not, say in one sentence which of your assumptions was wrong.

In [ ]:
# Check A: magnitude: mask the sentinel, convert the slip, then average
Tc = np.where(T > 10, T / 1000, T)   # ms -> s
good = Tc != -999
gs = []
for Lv in np.unique(L):
    m = (L == Lv) & good
    Tm = Tc[m].mean()
    g = 4 * math.pi**2 * Lv / Tm**2  # m/s^2
    gs.append(g)
    print(f"L = {Lv:.2f} m: n = {m.sum()}, <T> = {Tm:.4f} s, g = {g:.3f} m/s^2")
print(f"g = {np.mean(gs):.3f} m/s^2; spread = {max(gs) - min(gs):.3f} m/s^2")
# Check B: units and physics: 4 pi^2 L / T^2 is m/s^2 and must be close to 9.81
assert 9.5 < np.mean(gs) < 10.1, "g must be close to 9.81 m/s^2"
assert max(gs) - min(gs) < 0.2, "the five lengths must agree"

**Your turn.** Suppose the missing period had been logged as 0 instead of −999. What g does the 0.80 m length give if nobody masks it? Why is a 0 more dangerous than −999?

*Hint:* average the four values with 0 included; compare with 9.8 m/s².

In [ ]:
# Tc (periods in s, units fixed) and L (lengths in m) from above; the missing value sits where T == -999
# TODO: write your answer here

## Before you close the tab
- **Runtime → Restart session and run all**: every cell must run in order without errors (the Problem 4 `AssertionError` is meant to appear as text).
- **File → Save**; check the notebook is in your Drive.
- Your eight predictions: count how many were right and bring the number to Week 2.
- Mistakes we met: 10 µs lost in a JD (P1), `t += dt` (P2), uint8 wrap (P3), naive variance (P4), h = 10⁻⁸ reused (P5), "a list costs what an array costs" (P6), 21 crossings (P7), −999 averaged (P8).
- Each has a check: MJD grid, t = n·dt, int64 sum, σ² ≥ 0, h_opt from the model, `getsizeof`, plateau, physics range.
- Record your timing ratios (list vs array, row vs column) and bring them to the next session.